# Sentiment Analysis
Multilingual sentiment classification of YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Setup
Import dependencies: `pandas` for data, `transformers.pipeline` for
the model, `matplotlib`/`seaborn` for visualization.

In [14]:
import pandas as pd
import numpy as np
import os
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import torch

plt.rcParams["figure.dpi"] = 150

## Configuration
Batch selector. Change `BATCH_NAME` to switch between
`youtube_comments_batch_{1,2,3}.csv`.

In [10]:
# --- CONFIG ---
BASE_DIR = "../../01-preprocessing/output"
BATCH_NAME = "youtube_comments_batch_1.csv"
BATCH_PATH = os.path.join(BASE_DIR, BATCH_NAME)

## Load Batch
Read the preprocessed CSV. Each row is a cleaned comment with
`video_id`, `text`, `create_time`, `like_count`, `is_shouting`.

In [11]:
batch_df = pd.read_csv(BATCH_PATH)
print(f"Loaded {len(batch_df)} comments from {BATCH_NAME}")
batch_df.head()

Loaded 3392 comments from youtube_comments_batch_1.csv


,video_id,text,like_count,create_time,is_shouting
0,UKKUm9bQTTY,di pa siguro na brief si kalbo kung ano ang sa...,0,2026-06-16 00:00:00+00:00,0
1,UKKUm9bQTTY,throw this bloody baldwin in prison asap !! th...,0,2026-06-16 00:00:00+00:00,1
2,UKKUm9bQTTY,coach na malupit sa mga batang player,0,2026-06-16 00:00:00+00:00,1
3,UKKUm9bQTTY,malamang naka labas na ng bansa yan si bokal b...,1,2026-06-15 00:00:00+00:00,0
4,UKKUm9bQTTY,the killet ateneo and coach tab,1,2026-06-15 00:00:00+00:00,0


## Exploratory Data Analysis
Summary stats: videos in batch, date range, proportion
of all-caps comments, text length distribution.

In [12]:
print(f"Videos in batch: {batch_df['video_id'].nunique()}")
print(f"Date range: {batch_df['create_time'].min()} to {batch_df['create_time'].max()}")
print(f"Shouting comments: {batch_df['is_shouting'].sum()} ({batch_df['is_shouting'].mean():.1%})")
print(f"Text length stats:")
print(batch_df['text'].str.len().describe().to_string())

Videos in batch: 10
Date range: 2026-06-09 00:00:00+00:00 to 2026-06-18 05:47:00+00:00
Shouting comments: 124 (3.7%)
Text length stats:
count    3392.000000
mean      115.567512
std       139.836468
min         1.000000
25%        43.000000
50%        78.000000
75%       143.000000
max      3525.000000


## Model Loading
Load `cardiffnlp/twitter-xlm-roberta-base-sentiment` via Hugging Face
`pipeline`. Multilingual XLM-RoBERTa fine-tuned for social media
sentiment (positive/negative/neutral).

In [17]:
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# 1. Initial hardware check
if torch.cuda.is_available():
    device_setting = 0
    print(f"GPU detected: {torch.cuda.get_device_name(0)}. Attempting to initialize...")
else:
    device_setting = -1
    print("No GPU detected. Defaulting to CPU.")

# 2. Try loading on the selected device, fallback to CPU if incompatible
try:
    sentiment_pipeline = pipeline(
        "sentiment-analysis", 
        model=MODEL_NAME, 
        truncation=True,
        device=device_setting
    )
    # Test the pipeline with a dummy string to ensure the GPU kernel actually works
    if device_setting == 0:
        sentiment_pipeline("Sanity check")
    print("Successfully initialized pipeline!")
    
except Exception as e:
    # If a CUDA initialization or kernel error occurs, catch it and force CPU
    print(f"\n[WARNING] GPU initialization failed due to compatibility issues.")
    print(f"Error details: {e}")
    print("Falling back to CPU execution safely...\n")
    
    sentiment_pipeline = pipeline(
        "sentiment-analysis", 
        model=MODEL_NAME, 
        truncation=True,
        device=-1
    )
    print("Successfully initialized pipeline on CPU.")

GPU detected: NVIDIA GeForce GTX 1050 Ti. Attempting to initialize...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13915.24it/s]



[WARNING] GPU initialization failed due to compatibility issues.
Error details: CUDA out of memory. Tried to allocate 734.00 MiB. GPU 0 has a total capacity of 3.93 GiB of which 512.00 MiB is free. Including non-PyTorch memory, this process has 2.20 GiB memory in use. Of the allocated memory 2.07 GiB is allocated by PyTorch, and 85.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
Falling back to CPU execution safely...



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18373.22it/s]


Successfully initialized pipeline on CPU.


## Inference
Run the model on all comments. `batch_size=32` balances speed
and memory. Output: `sentiment_label` (POSITIVE/NEGATIVE/NEUTRAL)
and `sentiment_score` (model confidence, 0-1).

In [18]:
results = sentiment_pipeline(batch_df["text"].tolist(), batch_size=32)
batch_df["sentiment_label"] = [r["label"] for r in results]
batch_df["sentiment_score"] = [r["score"] for r in results]
batch_df[["text", "sentiment_label", "sentiment_score"]].head(10)

RuntimeError: index 514 is out of bounds for dimension 1 with size 514

## Per-Video Aggregation
Group scored comments by `video_id` to compare sentiment
across news sources covering the same crisis event.

In [ ]:
video_summary = batch_df.groupby("video_id").agg(
    total_comments=("text", "count"),
    mean_confidence=("sentiment_score", "mean"),
    pct_positive=("sentiment_label", lambda x: (x == "POSITIVE").mean() * 100),
    pct_negative=("sentiment_label", lambda x: (x == "NEGATIVE").mean() * 100),
    pct_neutral=("sentiment_label", lambda x: (x == "NEUTRAL").mean() * 100),
).round(1).sort_values("total_comments", ascending=False)

video_summary

## Shouting Analysis
Compare sentiment distributions between all-caps (shouting)
and normal comments. Tests whether angry expression correlates
with model-detected negative sentiment.

In [ ]:
shouting_stats = (
    batch_df.groupby("is_shouting")["sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("percentage")
    .reset_index()
)
shouting_stats

In [ ]:
shouting_contrast = batch_df.groupby("is_shouting").agg(
    count=("text", "count"),
    mean_confidence=("sentiment_score", "mean"),
    pct_positive=("sentiment_label", lambda x: (x == "POSITIVE").mean() * 100),
    pct_negative=("sentiment_label", lambda x: (x == "NEGATIVE").mean() * 100),
).round(1)
shouting_contrast

## Visualizations
Three plots summarizing the batch:
1. **Stacked bar** — sentiment % per video (which sources skew positive vs negative)
2. **Pie chart** — overall batch sentiment balance
3. **Shouting contrast** — whether all-caps comments are more negative (expected signal of anger/frustration)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Sentiment distribution per video (stacked bar)
videos_pivot = batch_df.groupby(["video_id", "sentiment_label"]).size().unstack(fill_value=0)
videos_pct = videos_pivot.div(videos_pivot.sum(axis=1), axis=0) * 100
videos_pct.plot(kind="bar", stacked=True, ax=axes[0], colormap="RdYlGn")
axes[0].set_title("Sentiment % per Video")
axes[0].set_ylabel("Percentage")
axes[0].legend(loc="upper right", fontsize=8)

# 2) Overall sentiment pie
overall_counts = batch_df["sentiment_label"].value_counts()
colors = {"POSITIVE": "#2ecc71", "NEGATIVE": "#e74c3c", "NEUTRAL": "#95a5a6"}
axes[1].pie(
    overall_counts.values,
    labels=overall_counts.index,
    autopct="%1.1f%%",
    colors=[colors.get(l, "#999") for l in overall_counts.index],
    startangle=90,
)
axes[1].set_title("Overall Sentiment")

# 3) Shouting vs normal contrast
shouting_plot = batch_df.groupby("is_shouting")["sentiment_label"].value_counts(normalize=True).mul(100).unstack()
shouting_plot.plot(kind="bar", ax=axes[2], colormap="RdYlGn", edgecolor="white")
axes[2].set_title("Sentiment: Shouting vs Normal")
axes[2].set_ylabel("Percentage")
axes[2].set_xticklabels(["Normal", "Shouting"], rotation=0)
axes[2].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## Export
Save the scored batch with sentiment labels and confidence scores
for downstream analysis or comparison across batches.

In [ ]:
output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"scored_{BATCH_NAME}")
batch_df.to_csv(output_path, index=False)
print(f"Exported {len(batch_df)} scored comments to {output_path}")